In [1]:
!pip install segmentation-models-pytorch timm opencv-python pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01


In [2]:
!pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128


In [2]:
!pip install --no-cache-dir torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu118 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 217.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 338.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 294.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 306.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 296.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 246.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 249.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 248.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 222.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 264.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 256.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, BatchSampler
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn
import albumentations as A
from typing import Tuple, Optional, List, Dict, Any
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

In [6]:
torch.load("/kaggle/input/models/meowmeowmeeowww/regular-2350-effnet-deeplab/tensorflow2/default/1/best_regular(1).pth", map_location="cpu").get('val_dice', 'Not found')

0.8983163070678711

In [5]:
import torch.nn.functional as F
from tqdm import tqdm

# =========================
# CONFIGURATION
# =========================
UNLABELLED_IMAGES_DIR = Path("/kaggle/input/competitions/dl-lab-3-product-segmentation/unlabeled/images")
OUTPUT_MASKS_DIR = Path("/kaggle/working/pseudo_masks")
OUTPUT_MASKS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = Path("/kaggle/input/models/meowmeowmeeowww/deeplab-effnetb4/tensorflow2/default/1/distilled_best_fixed.pth")
IMG_SIZE = 352
THRESHOLD = 0.54
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENCODER_NAME = "timm-efficientnet-b4"
ENCODER_WEIGHTS = "noisy-student"

# =========================
# LOAD MODEL
# =========================
def build_model():
    model = smp.DeepLabV3Plus(
        encoder_name=ENCODER_NAME,
        encoder_weights=None,
        encoder_output_stride=8,
        decoder_channels=256,
        decoder_atrous_rates=(12, 24, 36),
        in_channels=3,
        classes=1,
        activation=None,
        upsampling=4,
    )
    return model

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
model = build_model()
state_dict = checkpoint["model_state_dict"]
if list(state_dict.keys())[0].startswith("module."):
    state_dict = {k[7:]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

preprocess_fn = get_preprocessing_fn(ENCODER_NAME, pretrained=ENCODER_WEIGHTS)

# =========================
# TTA + POSTPROCESSING
# =========================
def postprocess_mask(prob, threshold=THRESHOLD, min_area=200):
    """Clean mask with morphological closing and small object removal."""
    mask = (prob > threshold).astype(np.uint8)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    filtered = np.zeros_like(mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            filtered[labels == i] = 1
    return filtered

@torch.no_grad()
def predict_tta(image_rgb, scales=(0.75, 1.0, 1.25), flips=True):
    """Multi-scale TTA with horizontal/vertical flips."""
    h, w = image_rgb.shape[:2]
    preds = []
    for scale in scales:
        new_size = int(IMG_SIZE * scale)
        inp = cv2.resize(image_rgb, (new_size, new_size), interpolation=cv2.INTER_LINEAR)
        inp = preprocess_fn(inp.astype(np.float32))
        inp_tensor = torch.from_numpy(inp.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

        # Original
        logits = model(inp_tensor)
        prob = torch.sigmoid(logits)
        prob = F.interpolate(prob, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        preds.append(prob)

        if flips:
            # Horizontal flip
            logits_h = model(torch.flip(inp_tensor, dims=[-1]))
            prob_h = torch.flip(torch.sigmoid(logits_h), dims=[-1])
            prob_h = F.interpolate(prob_h, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
            preds.append(prob_h)

            # Vertical flip
            logits_v = model(torch.flip(inp_tensor, dims=[-2]))
            prob_v = torch.flip(torch.sigmoid(logits_v), dims=[-2])
            prob_v = F.interpolate(prob_v, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
            preds.append(prob_v)

    # Average all predictions
    final_prob = torch.stack(preds).mean(dim=0)[0, 0].cpu().numpy()
    final_prob = cv2.resize(final_prob, (w, h), interpolation=cv2.INTER_LINEAR)
    return final_prob

# =========================
# GENERATE PSEUDO-LABELS
# =========================
image_paths = sorted([p for p in UNLABELLED_IMAGES_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
print(f"Found {len(image_paths)} unlabelled images")

for img_path in tqdm(image_paths, desc="Pseudo-labeling"):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # TTA prediction
    prob = predict_tta(img_rgb, scales=(0.75, 1.0, 1.25), flips=True)

    # Postprocess and save
    mask = postprocess_mask(prob, threshold=THRESHOLD, min_area=200)
    mask_path = OUTPUT_MASKS_DIR / f"{img_path.stem}.png"
    cv2.imwrite(str(mask_path), mask * 255)

print(f"Pseudo-masks saved to {OUTPUT_MASKS_DIR}")

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

Found 350 unlabelled images


Pseudo-labeling: 100%|██████████| 350/350 [03:23<00:00,  1.72it/s]

Pseudo-masks saved to /kaggle/working/pseudo_masks


In [14]:
!pip install transformers -q

In [15]:
from transformers import SegformerForSemanticSegmentation

In [19]:
# =========================
# CONFIG
# =========================
DATA_ROOT = Path(
    r"/kaggle/input/competitions/dl-lab-3-product-segmentation/train"
)
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
PSEUDO_IMAGES_DIR = Path(r"/kaggle/input/competitions/dl-lab-3-product-segmentation/unlabeled/images")
PSEUDO_MASKS_DIR = Path(r"/kaggle/working/pseudo_masks")
SAVE_DIR = Path("./seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MULTI_SCALE_SIZES = [288, 320, 352, 384]
VAL_SIZE = 352
BATCH_SIZE = 4
NUM_EPOCHS = 35
SWA_START = 22
LR = 6e-5
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.2
NUM_WORKERS = 4
SEED = 42
THRESHOLD = 0.54

MODEL_NAME = "SegFormer"
ENCODER_NAME = "mit_b2"
ENCODER_WEIGHTS = "imagenet"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# UTILS
# =========================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def dice_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()
    
    preds_flat = preds.view(preds.size(0), -1)
    targets_flat = targets.view(targets.size(0), -1)
    
    intersection = (preds_flat * targets_flat).sum(dim=1)
    denom = preds_flat.sum(dim=1) + targets_flat.sum(dim=1)
    
    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean().item()


def iou_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()
    
    preds_flat = preds.view(preds.size(0), -1)
    targets_flat = targets.view(targets.size(0), -1)
    
    intersection = (preds_flat * targets_flat).sum(dim=1)
    union = preds_flat.sum(dim=1) + targets_flat.sum(dim=1) - intersection
    
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


# =========================
# DATASET
# =========================

def get_train_augs():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        A.Affine(
            translate_percent=(-0.03, 0.03),
            scale=(0.95, 1.05),
            rotate=(-5, 5),
            shear=(-2, 2),
            p=0.5,
        ),
        A.OneOf([
            A.RandomBrightnessContrast(0.3, 0.3),
            A.CLAHE(),
            A.RandomGamma(),
        ], p=0.6),
        A.HueSaturationValue(5, 10, 5, p=0.3),
        A.GaussNoise(std_range=(0.01, 0.03), p=0.2), 
        
        A.GaussianBlur(blur_limit=3, p=0.2),         
        A.CoarseDropout(
            num_holes_range=(1, 3),
            hole_height_range=(0.05, 0.1),
            hole_width_range=(0.05, 0.1),
            p=0.1,
        ),
    ])


def get_val_augs():
    return A.Compose([])


class BatchSizeBatchSampler(BatchSampler):
    """BatchSampler that selects a random size for each batch"""
    def __init__(self, sampler, batch_size, multi_scale_sizes, drop_last):
        super().__init__(sampler, batch_size, drop_last)
        self.multi_scale_sizes = multi_scale_sizes
        self.current_scale = None
        self.dataset = None
        
    def __iter__(self):
        for batch in super().__iter__():
            self.current_scale = random.choice(self.multi_scale_sizes)
            # Set current scale on dataset for access in __getitem__
            if self.dataset is not None:
                self.dataset._current_scale = self.current_scale
            yield batch


class BinarySegDataset2(Dataset):
    """
    Segmentation dataset with optional Copy-Paste augmentation using a precomputed object database.
    
    Args:
        images_dir: Path to directory containing input images.
        masks_dir: Path to directory containing ground truth masks (PNG).
        img_size: Target image size (square).
        encoder_name: Segmentation model encoder name (e.g., 'efficientnet-b3').
        encoder_weights: Pretrained weights ('imagenet' or None).
        transforms: Albumentations transform pipeline.
        copy_paste_prob: Probability of applying Copy-Paste augmentation (0.0 = disabled).
        cache_images: If True, load all images into RAM for faster access.
        min_object_area: Minimum area (pixels) for an object to be stored in the database.
        max_objects_per_image: Maximum number of objects to extract per image (memory control).
        multi_scale_sizes: List of sizes for multi-scale training (None for fixed size).
    """
    
    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        img_size: int = 352,
        encoder_name: str = "resnet34",
        encoder_weights: Optional[str] = "imagenet",
        transforms=None,
        copy_paste_prob: float = 0.0,
        cache_images: bool = False,
        min_object_area: int = 50,
        max_objects_per_image: int = 10,
        multi_scale_sizes: Optional[List[int]] = None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.transforms = transforms
        self.copy_paste_prob = copy_paste_prob
        self.cache_images = cache_images
        self.min_object_area = min_object_area
        self.max_objects_per_image = max_objects_per_image
        self.multi_scale_sizes = multi_scale_sizes
        self._current_scale = None  # Will be set by BatchSampler
        
        # Preprocessing function for encoder normalization
        self.preprocess_input = None
        if encoder_weights is not None:
            self.preprocess_input = get_preprocessing_fn(encoder_name, pretrained=encoder_weights)
        
        # Build list of (image_path, mask_path) pairs
        self.samples = []
        for mask_path in sorted(self.masks_dir.glob("*.png")):
            stem = mask_path.stem
            image_path = self._find_image_for_stem(stem)
            if image_path is not None:
                self.samples.append((image_path, mask_path))
        
        if not self.samples:
            raise RuntimeError(f"No paired image/mask samples found in {images_dir} and {masks_dir}")
        
        print(f"Found {len(self.samples)} paired samples")
        
        # Cache for images (if enabled)
        self._cache: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}
        if self.cache_images:
            self._load_all_into_cache()
        
        # Object database for Copy-Paste (precomputed object crops)
        self._object_db: List[Tuple[np.ndarray, np.ndarray]] = []  # (obj_img, obj_mask)
        if self.copy_paste_prob > 0.0:
            self._build_object_database()
    
    def _find_image_for_stem(self, stem: str) -> Optional[Path]:
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
            p = self.images_dir / f"{stem}{ext}"
            if p.exists():
                return p
            # Try lowercase/uppercase variations if needed
            p_lower = self.images_dir / f"{stem.lower()}{ext}"
            if p_lower.exists():
                return p_lower
        return None
    
    def _load_all_into_cache(self):
        """Load all images and masks into RAM for faster access."""
        print("Pre-loading dataset into RAM cache...")
        for idx, (img_path, msk_path) in enumerate(self.samples):
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            msk = cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE)
            msk = (msk > 0).astype(np.uint8)
            self._cache[idx] = (img, msk)
        print(f"Cached {len(self._cache)} images/masks")
    
    def _load_sample(self, idx: int, return_uint8_mask: bool = False) -> Tuple[np.ndarray, np.ndarray]:
        # Check cache first
        if self.cache_images and idx in self._cache:
            img, msk_uint8 = self._cache[idx]
            img = img.copy()
            if return_uint8_mask:
                return img, msk_uint8.copy()
            else:
                return img, msk_uint8.astype(np.float32)
        
        # Load from disk
        img_path, msk_path = self.samples[idx]
        img = cv2.imread(str(img_path))
        if img is None:
            raise RuntimeError(f"Cannot read image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        msk = cv2.imread(str(msk_path), cv2.IMREAD_GRAYSCALE)
        if msk is None:
            raise RuntimeError(f"Cannot read mask: {msk_path}")
        
        if return_uint8_mask:
            msk = (msk > 0).astype(np.uint8)
        else:
            msk = (msk > 0).astype(np.float32)
        
        return img, msk
    
    def _build_object_database(self):
        print("Building object database for Copy-Paste...")
        total_objects = 0
        
        for idx in range(len(self.samples)):
            # Load full image and mask as uint8
            img, mask_uint8 = self._load_sample(idx, return_uint8_mask=True)
            
            # Find connected components (objects) in the mask
            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
                mask_uint8, connectivity=8
            )
            
            objects_in_this_image = 0
            for obj_id in range(1, num_labels):  # skip background (0)
                area = stats[obj_id, cv2.CC_STAT_AREA]
                if area < self.min_object_area:
                    continue
                
                # Bounding box
                x = stats[obj_id, cv2.CC_STAT_LEFT]
                y = stats[obj_id, cv2.CC_STAT_TOP]
                w = stats[obj_id, cv2.CC_STAT_WIDTH]
                h = stats[obj_id, cv2.CC_STAT_HEIGHT]
                
                # Crop object image and mask
                obj_img = img[y:y+h, x:x+w].copy()
                obj_mask = (labels[y:y+h, x:x+w] == obj_id).astype(np.uint8)
                
                # Store in database
                self._object_db.append((obj_img, obj_mask))
                total_objects += 1
                objects_in_this_image += 1
                
                # Limit per image to avoid excessive memory
                if objects_in_this_image >= self.max_objects_per_image:
                    break
        
        print(f"Precomputed {total_objects} objects for Copy-Paste from {len(self.samples)} images")
        if total_objects == 0:
            print("Warning: No objects found for Copy-Paste. Augmentation will be disabled.")
            self.copy_paste_prob = 0.0
    
    def _copy_paste(self, image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Paste a random precomputed object into the image and mask.
        
        Args:
            image: (H, W, 3) uint8 image.
            mask: (H, W) uint8 mask (0/1).
        
        Returns:
            Augmented (image, mask).
        """
        if not self._object_db:
            return image, mask
        
        # Randomly select an object from the database
        obj_img, obj_mask = random.choice(self._object_db)
        obj_h, obj_w = obj_img.shape[:2]
        img_h, img_w = image.shape[:2]
        
        # Skip if object is larger than image (should not happen with proper resizing, but safe)
        if obj_h >= img_h or obj_w >= img_w:
            return image, mask
        
        # Random paste location
        py = random.randint(0, img_h - obj_h)
        px = random.randint(0, img_w - obj_w)
        
        # Paste object image (only where object mask is 1)
        roi_img = image[py:py+obj_h, px:px+obj_w]
        roi_img[obj_mask == 1] = obj_img[obj_mask == 1]
        image[py:py+obj_h, px:px+obj_w] = roi_img
        
        # Update mask
        mask[py:py+obj_h, px:px+obj_w][obj_mask == 1] = 1
        
        return image, mask
    
    def __len__(self) -> int:
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        # Load primary sample
        image, mask = self._load_sample(idx, return_uint8_mask=False)
    
        # Copy-Paste augmentation (if enabled)
        if self.copy_paste_prob > 0.0 and random.random() < self.copy_paste_prob:
            mask_uint8 = (mask > 0).astype(np.uint8)
            image, mask_uint8 = self._copy_paste(image, mask_uint8)
            mask = mask_uint8.astype(np.float32)
    
        # Determine target size
        # For multi-scale training, use the scale set by BatchSampler
        if self.multi_scale_sizes is not None and self._current_scale is not None:
            target_size = self._current_scale
        else:
            target_size = self.img_size
    
        # Apply resize
        resize_aug = A.Compose([
            A.LongestMaxSize(max_size=target_size),
            A.PadIfNeeded(target_size, target_size, border_mode=0),
        ])
        augmented = resize_aug(image=image, mask=mask)
        image = augmented["image"]
        mask = augmented["mask"]
    
        # Apply other transforms (augmentations for train, empty for val)
        if self.transforms is not None:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]
    
        # Convert to tensor and apply encoder preprocessing
        if isinstance(image, np.ndarray):
            if self.preprocess_input is not None:
                image = self.preprocess_input(image)
            else:
                image = image / 255.0
            image = torch.from_numpy(image.transpose(2, 0, 1)).float()
    
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).unsqueeze(0).float()
        elif hasattr(mask, 'dim') and mask.dim() == 2:
            mask = mask.unsqueeze(0).float()
    
        return image, mask


# =========================
# MODEL
# =========================
def build_model() -> nn.Module:
    if MODEL_NAME == "Unet":
        model = smp.Unet(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "FPN":
        model = smp.FPN(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "DeepLabV3Plus":
        model = smp.DeepLabV3Plus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            encoder_output_stride=8,
            decoder_channels=256,
            decoder_atrous_rates=(12, 24, 36),
            in_channels=3,
            classes=1,
            activation=None,
            upsampling=4,
        )
    elif MODEL_NAME == "SegFormer":
        model = smp.MAnet(
            encoder_name=ENCODER_NAME,  # "mit_b2"
            encoder_weights=ENCODER_WEIGHTS,
            decoder_channels=(256, 128, 64, 32, 16),
            decoder_use_batchnorm=True,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")
    return model


# =========================
# TRAIN / VAL LOOPS
# =========================
def train_one_epoch(model, loader, optimizer, loss_fn, device, scheduler=None):
    model.train()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    scaler = torch.amp.GradScaler('cuda') if device == "cuda" else None
    
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        
        if device == "cuda":
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = loss_fn(logits, masks)
        else:
            logits = model(images)
            loss = loss_fn(logits, masks)
        
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        #if scheduler is not None:
        #    scheduler.step()


        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits.detach(), masks)
        running_iou += iou_score_from_logits(logits.detach(), masks)
        

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }


@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = loss_fn(logits, masks)

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits, masks)
        running_iou += iou_score_from_logits(logits, masks)

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }


# =========================
# MAIN
# =========================
def main():
    seed_everything(SEED)
    
    # Create datasets
    train_dataset = BinarySegDataset2(
        IMAGES_DIR, MASKS_DIR,
        img_size=VAL_SIZE,  # Fallback size (not used with multi-scale)
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        transforms=get_train_augs(),
        copy_paste_prob=0.2,
        cache_images=True,
        min_object_area=50,
        max_objects_per_image=5,
        multi_scale_sizes=MULTI_SCALE_SIZES,  # Enable multi-scale
    )

    pseudo_dataset = BinarySegDataset2(
        PSEUDO_IMAGES_DIR, PSEUDO_MASKS_DIR,
        img_size=VAL_SIZE,
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        transforms=get_train_augs(),
        copy_paste_prob=0.05,          # very low – pseudo masks may be imperfect
        cache_images=True,
        min_object_area=50,
        max_objects_per_image=5,
        multi_scale_sizes=MULTI_SCALE_SIZES,
    )
    
    val_dataset = BinarySegDataset2(
        IMAGES_DIR, MASKS_DIR,
        img_size=VAL_SIZE,  # Fixed size for validation
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        transforms=get_val_augs(),
        copy_paste_prob=0.0,
        cache_images=False,
        multi_scale_sizes=None,  # No multi-scale for validation
    )
    
    # Split indices
    indices = list(range(len(train_dataset)))
    random.shuffle(indices)
    split = int(len(indices) * VAL_RATIO)
    
    train_orig_indices = indices[split:]
    val_indices = indices[:split]

    train_orig_subset = torch.utils.data.Subset(train_dataset, train_orig_indices)
    val_subset = torch.utils.data.Subset(val_dataset, val_indices)
    combined_train_dataset = torch.utils.data.ConcatDataset([train_orig_subset, pseudo_dataset])
    
    print(f"Train samples: {len(combined_train_dataset)} (original: {len(train_orig_subset)} + pseudo: {len(pseudo_dataset)})")
    print(f"Val samples: {len(val_subset)}")

    # =========================
    # Custom Collate for Multi-Scale + ConcatDataset
    # =========================
    
    def create_multiscale_collate(datasets, multi_scale_sizes):
        """Create a collate function that ensures all samples in a batch have the same size."""
        
        def multiscale_collate(batch):
            # Randomly select a scale for this batch
            scale = random.choice(multi_scale_sizes)
            
            # Process each sample to ensure consistent size
            processed_batch = []
            resize_transform = A.Compose([
                A.LongestMaxSize(max_size=scale),
                A.PadIfNeeded(scale, scale, border_mode=0),
            ])
            
            for image, mask in batch:
                # Convert tensor to numpy for albumentations
                img_np = image.permute(1, 2, 0).cpu().numpy()
                msk_np = mask.squeeze(0).cpu().numpy()
                
                # Resize to batch scale
                augmented = resize_transform(image=img_np, mask=msk_np)
                
                # Convert back to tensor
                img_resized = torch.from_numpy(augmented['image'].transpose(2, 0, 1)).float()
                msk_resized = torch.from_numpy(augmented['mask']).unsqueeze(0).float()
                
                processed_batch.append((img_resized, msk_resized))
            
            return torch.utils.data.default_collate(processed_batch)
        
        return multiscale_collate
    
    # Create the collate function
    collate_fn = create_multiscale_collate([train_dataset, pseudo_dataset], MULTI_SCALE_SIZES)
    
    # Replace the DataLoader creation:
    train_loader = DataLoader(
        combined_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,  # Can use shuffle now since we don't need BatchSampler
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE == "cuda" else False,
        drop_last=False,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE == "cuda" else False,
        drop_last=False,
    )

    # Loss functions
    lovasz = smp.losses.LovaszLoss(mode="binary", per_image=True)
    dice = smp.losses.DiceLoss(mode="binary", from_logits=True)
    def loss_fn(logits, targets):
        return 0.6 * lovasz(logits, targets) + 0.4 * dice(logits, targets)

    model = build_model().to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=7, T_mult=2, eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=NUM_EPOCHS,
    eta_min=1e-6
    )
    
    swa_model = AveragedModel(model)
    swa_scheduler = SWALR(optimizer, swa_lr=LR/20, anneal_epochs=5)

    best_val_dice = -1.0
    best_swa_dice = -1.0 
    history = []

    print(f"Training with multi-scale sizes: {MULTI_SCALE_SIZES}")
    print(f"Validation size: {VAL_SIZE}")
    def get_aug_factor(epoch):
        if epoch <= 20:
            return 1.00
        elif epoch <= 27:
            return 0.95
        elif epoch <= 32:
            return 0.9
        else:
            return 0.85


    def aug_scheduler(dataset, factor):
        if hasattr(dataset, 'copy_paste_prob'):
            dataset.copy_paste_prob = 0.2 * factor
    
        # Helper to recursively scale probabilities
        def scale_transform(transform):
            # Handle OneOf (which contains a list of transforms)
            if isinstance(transform, A.OneOf):
                for sub_t in transform.transforms:
                    scale_transform(sub_t)
            # Handle Compose (though usually top-level)
            elif isinstance(transform, A.Compose):
                for sub_t in transform.transforms:
                    scale_transform(sub_t)
            # Handle individual augmentations
            if hasattr(transform, 'p'):
                # Store original probability if not already stored
                if not hasattr(transform, '_original_p'):
                    transform._original_p = transform.p
                # Apply scaling
                transform.p = transform._original_p * factor
    
        # Apply to all transforms in the dataset's Compose pipeline
        for t in dataset.transforms.transforms:
            scale_transform(t)
    
    
    for epoch in range(1, NUM_EPOCHS + 1):
        aug_factor = get_aug_factor(epoch)
        aug_scheduler(train_dataset, aug_factor)
        aug_scheduler(pseudo_dataset, aug_factor)
        
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)

        if epoch >= SWA_START:
            swa_model.update_parameters(model)
            swa_scheduler.step()
            scheduler.step()
            
            val_metrics = validate_one_epoch(swa_model, val_loader, loss_fn, DEVICE)
            
            if val_metrics["dice"] > best_swa_dice:
                best_swa_dice = val_metrics["dice"]
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": swa_model.state_dict(),
                    "val_dice": best_swa_dice,
                    "type": "swa",
                }, SAVE_DIR / "best_swa.pth")
                print(f"Saved new best SWA model with val_dice={best_swa_dice:.4f}")
        else:
            scheduler.step()
            
            # Validate with regular model
            val_metrics = validate_one_epoch(model, val_loader, loss_fn, DEVICE)
            
            if val_metrics["dice"] > best_val_dice:
                best_val_dice = val_metrics["dice"]
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_dice": best_val_dice,
                    "type": "regular",
                }, SAVE_DIR / "best_regular.pth")
                print(f"Saved new best regular model with val_dice={best_val_dice:.4f}")
                
        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_dice": train_metrics["dice"],
            "train_iou": train_metrics["iou"],
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "val_iou": val_metrics["iou"],
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
            f"train_loss={row['train_loss']:.4f} train_dice={row['train_dice']:.4f} train_iou={row['train_iou']:.4f} | "
            f"val_loss={row['val_loss']:.4f} val_dice={row['val_dice']:.4f} val_iou={row['val_iou']:.4f}"
        )
        
        # Save last
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": row["val_dice"],
                "config": {
                    "MODEL_NAME": MODEL_NAME,
                    "ENCODER_NAME": ENCODER_NAME,
                    "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                    "VAL_SIZE": VAL_SIZE,
                    "MULTI_SCALE_SIZES": MULTI_SCALE_SIZES,
                },
            },
            SAVE_DIR / "last.pth",
        )


    print("Updating BatchNorm statistics for SWA model...")
    swa_model.train()
    update_bn(train_loader, swa_model, device=DEVICE)
    swa_model.eval()      
    final_metrics = validate_one_epoch(swa_model, val_loader, loss_fn, DEVICE)
    print(f"Final SWA model after BN update: val_dice={final_metrics['dice']:.4f}")
    
    # Save final SWA model
    torch.save(
        {
            "model_state_dict": swa_model.state_dict(),
            "val_dice": final_metrics["dice"],
            "type": "swa_final",
            "config": {
                "MODEL_NAME": MODEL_NAME,
                "ENCODER_NAME": ENCODER_NAME,
                "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                "VAL_SIZE": VAL_SIZE,
                "MULTI_SCALE_SIZES": MULTI_SCALE_SIZES,
        },
    }, SAVE_DIR / "swa_final.pth")

    # Save history
    history_df = pd.DataFrame(history)
    history_df.to_csv(SAVE_DIR / "history.csv", index=False)
    print(f"Training finished. Best val_dice={best_val_dice:.4f}")


if __name__ == "__main__":
    main()

Found 2000 paired samples
Pre-loading dataset into RAM cache...
Cached 2000 images/masks
Building object database for Copy-Paste...
Precomputed 2379 objects for Copy-Paste from 2000 images
Found 350 paired samples
Pre-loading dataset into RAM cache...
Cached 350 images/masks
Building object database for Copy-Paste...
Precomputed 414 objects for Copy-Paste from 350 images
Found 2000 paired samples
Train samples: 1950 (original: 1600 + pseudo: 350)
Val samples: 400
Training with multi-scale sizes: [288, 320, 352, 384]
Validation size: 352
Saved new best regular model with val_dice=0.8031
Epoch 001/35 | train_loss=0.7280 train_dice=0.7131 train_iou=0.6122 | val_loss=0.5657 val_dice=0.8031 val_iou=0.7189
Saved new best regular model with val_dice=0.8651
Epoch 002/35 | train_loss=0.4665 train_dice=0.8537 train_iou=0.7736 | val_loss=0.4163 val_dice=0.8651 val_iou=0.7893
Saved new best regular model with val_dice=0.8795
Epoch 003/35 | train_loss=0.3751 train_dice=0.8827 train_iou=0.8104 | val